## DurianAI — Vision CNN Training

> Phase 1: MobileNetV2 Transfer Learning → TFLite INT8 for browser deployment

**Data sources:**
- Roboflow durian-ripeness-detection-xtned (1,438 images, 3-class: Ripe/Unripe/Defect)
- **Roboflow durian_mutruity ★ NEW (3,000 images, 3-class: defective/immature/mature)**
- Zenodo Multi-Modal RGB images (189 samples, 3-class: Unripe/Ripe/Overripe)
- Rom1420 GitHub (4-class: Ripe1-4)

**Target:** 85%+ accuracy, ~4MB TFLite model

In [ ]:
#@title 1. Setup { display-mode: "form" }
!pip install -q roboflow scikit-learn tensorflow

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from pathlib import Path
from PIL import Image
from collections import Counter

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
#@title 2. Download Roboflow Dataset { display-mode: "form" }
ROBOFLOW_API_KEY = '' #@param {type:"string"}

DATA_DIR = '/content/durian_data'
os.makedirs(DATA_DIR, exist_ok=True)

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('durian-cnn').project('durian-ripeness-detection-xtned')
    dataset = project.versions()[0].download('folder', location=DATA_DIR)
    print('Roboflow dataset downloaded!')
else:
    print('No API key. Manual download instructions:')
    print('1. Go to https://universe.roboflow.com/durian-cnn/durian-ripeness-detection-xtned')
    print('2. Sign up (free) → Download Dataset → Folder format')
    print('3. Upload the ZIP to Colab and extract')
    # Check if uploaded
    roboflow_zip = '/content/roboflow_durian.zip'
    if os.path.exists(roboflow_zip):
        import zipfile
        with zipfile.ZipFile(roboflow_zip) as z:
            z.extractall(DATA_DIR)
        print('Extracted uploaded ZIP!')

In [ ]:
#@title 3. Download Zenodo RGB { display-mode: "form" }
import urllib.request, zipfile

# Zenodo RGB is 19 GB — download only if needed
download_zenodo_rgb = False #@param {type:"boolean"}

if download_zenodo_rgb:
    rgb_url = 'https://zenodo.org/records/18603796/files/dataset_clean_rgb.zip?download=1'
    rgb_zip = '/content/durian_data/zenodo_rgb.zip'
    print('Downloading Zenodo RGB (19 GB) — this will take a while...')
    urllib.request.urlretrieve(rgb_url, rgb_zip)
    print('Done!')
    rgb_extract = '/content/durian_data/zenodo_rgb'
    with zipfile.ZipFile(rgb_zip) as z:
        z.extractall(rgb_extract)
    print(f'Extracted to {rgb_extract}')

In [ ]:
#@title 2b. Download durian_mutruity Dataset (3,000 images) ★ { display-mode: "form" }
download_mutruity = True #@param {type:"boolean"}

MUTRUITY_DIR = '/content/durian_mutruity'

if download_mutruity and ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('wjy-tis6h').project('durian_mutruity')
    print("Downloading durian_mutruity (3,000 images, CC BY 4.0)...")
    dataset = project.versions()[0].download('folder', location=MUTRUITY_DIR)
    print("durian_mutruity downloaded!")
    for split in ["train", "valid", "test"]:
        d = os.path.join(MUTRUITY_DIR, split)
        if os.path.isdir(d):
            total = sum(len(files) for _, _, files in os.walk(d))
            print(f"  {split}: {total} files")
elif download_mutruity and not ROBOFLOW_API_KEY:
    print("Need API key. Manual download:")
    print("1. Go to https://universe.roboflow.com/wjy-tis6h/durian_mutruity")
    print("2. Sign up → Download Dataset → Folder format")
    print("3. Upload ZIP to Colab at /content/durian_mutruity/")
else:
    print("Skipping durian_mutruity download")


In [ ]:
#@title 4. Load & Prepare Data { display-mode: "form" }

IMG_SIZE = 224
BATCH_SIZE = 32

# Use Roboflow folder structure
# Expected: DATA_DIR/train/{Ripe,Unripe,Defect}/ etc.

# Unified label mapping
# Roboflow: Ripe → ripe, Unripe → unripe, Defect → defect
# Zenodo: Unripe → unripe, Ripe → ripe, Overripe → overripe
# Rom1420: Ripe1 → unripe, Ripe2 → ripe, Ripe3 → overripe, Ripe4 → overripe

UNIFIED_LABELS = ['unripe', 'ripe', 'overripe']  # Phase 1: 3-class
# Note: Roboflow 'Defect' will be excluded from 3-class classification

def load_image_folder(base_dir, split='train', label_mapping=None):
    X, y = [], []
    split_dir = os.path.join(base_dir, split)
    if not os.path.isdir(split_dir):
        split_dir = os.path.join(base_dir, 'valid' if split == 'val' else split)
    
    if not os.path.isdir(split_dir):
        return np.array(X), np.array(y)
    
    for cls_dir in os.listdir(split_dir):
        cls_path = os.path.join(split_dir, cls_dir)
        if not os.path.isdir(cls_path):
            continue
        mapped_label = cls_dir.lower()
        if label_mapping and cls_dir in label_mapping:
            mapped_label = label_mapping[cls_dir]
        # Skip defect class for 3-class
        if mapped_label not in UNIFIED_LABELS:
            continue
        for img_file in os.listdir(cls_path):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img_path = os.path.join(cls_path, img_file)
            try:
                img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                X.append(np.array(img, dtype=np.float32) / 255.0)
                y.append(mapped_label)
            except:
                pass
    return np.array(X), np.array(y)

# Load Roboflow data
X_train, y_train = load_image_folder(DATA_DIR, 'train',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})
X_val, y_val = load_image_folder(DATA_DIR, 'valid',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})
X_test, y_test = load_image_folder(DATA_DIR, 'test',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})

print(f'Roboflow: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
print(f'Train distribution: {dict(Counter(y_train))}')
print(f'Val distribution: {dict(Counter(y_val))}')

# Encode labels
label_map = {name: i for i, name in enumerate(UNIFIED_LABELS)}
y_train_enc = np.array([label_map[l] for l in y_train])
y_val_enc = np.array([label_map[l] for l in y_val])
y_test_enc = np.array([label_map[l] for l in y_test])
print(f'\nLabels: {UNIFIED_LABELS}')

In [ ]:
#@title 4. Load & Prepare Data { display-mode: "form" }

IMG_SIZE = 224
BATCH_SIZE = 32

UNIFIED_LABELS = ['unripe', 'ripe', 'overripe']  # Phase 1: 3-class

def load_image_folder(base_dir, split='train', label_mapping=None, exclude_classes=None):
    """Load images from folder structure: split/class/images"""
    X, y = [], []
    split_dir = os.path.join(base_dir, split)
    if not os.path.isdir(split_dir):
        split_dir = os.path.join(base_dir, 'valid' if split == 'val' else split)
    
    if not os.path.isdir(split_dir):
        return np.array(X), np.array(y)
    
    if exclude_classes is None:
        exclude_classes = []
    
    for cls_dir in os.listdir(split_dir):
        cls_path = os.path.join(split_dir, cls_dir)
        if not os.path.isdir(cls_path):
            continue
        mapped_label = cls_dir.lower()
        if label_mapping and cls_dir in label_mapping:
            mapped_label = label_mapping[cls_dir]
        # Skip excluded classes (e.g. Defect)
        if mapped_label not in UNIFIED_LABELS or mapped_label in exclude_classes:
            continue
        for img_file in os.listdir(cls_path):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img_path = os.path.join(cls_path, img_file)
            try:
                img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                X.append(np.array(img, dtype=np.float32) / 255.0)
                y.append(mapped_label)
            except:
                pass
    return np.array(X), np.array(y)

# --- Load Roboflow xtned (original) ---
X_train, y_train = load_image_folder(DATA_DIR, 'train',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})
X_val, y_val = load_image_folder(DATA_DIR, 'valid',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})
X_test, y_test = load_image_folder(DATA_DIR, 'test',
    label_mapping={'Ripe': 'ripe', 'Unripe': 'unripe', 'Defect': 'defect'})

print(f'xtned (original): Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')
print(f'  Train distribution: {dict(Counter(y_train))}')

# --- Load durian_mutruity (new, 3,000 images) ---
if os.path.isdir(MUTRUITY_DIR):
    X_m_train, y_m_train = load_image_folder(MUTRUITY_DIR, 'train',
        label_mapping={'immature': 'unripe', 'mature': 'ripe', 'defective': 'defect'})
    X_m_val, y_m_val = load_image_folder(MUTRUITY_DIR, 'valid',
        label_mapping={'immature': 'unripe', 'mature': 'ripe', 'defective': 'defect'})
    X_m_test, y_m_test = load_image_folder(MUTRUITY_DIR, 'test',
        label_mapping={'immature': 'unripe', 'mature': 'ripe', 'defective': 'defect'})
    
    print(f'\nmutruity (new): Train={len(X_m_train)}, Val={len(X_m_val)}, Test={len(X_m_test)}')
    print(f'  Train distribution: {dict(Counter(y_m_train))}')
    
    # Merge datasets (concatenate)
    if len(X_m_train) > 0:
        X_train = np.concatenate([X_train, X_m_train]) if len(X_train) > 0 else X_m_train
        y_train = np.concatenate([y_train, y_m_train]) if len(y_train) > 0 else y_m_train
    if len(X_m_val) > 0:
        X_val = np.concatenate([X_val, X_m_val]) if len(X_val) > 0 else X_m_val
        y_val = np.concatenate([y_val, y_m_val]) if len(y_val) > 0 else y_m_val
    if len(X_m_test) > 0:
        X_test = np.concatenate([X_test, X_m_test]) if len(X_test) > 0 else X_m_test
        y_test = np.concatenate([y_test, y_m_test]) if len(y_test) > 0 else y_m_test
    
    total = len(X_train) + len(X_val) + len(X_test)
    print(f'\n✅ Combined (xtned + mutruity): {total} images total')
    print(f'  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
    print(f'  Train distribution: {dict(Counter(y_train))}')
else:
    print('\nmutruity dataset not found. Using only xtned.')

# Encode labels
label_map = {name: i for i, name in enumerate(UNIFIED_LABELS)}
y_train_enc = np.array([label_map[l] for l in y_train])
y_val_enc = np.array([label_map[l] for l in y_val])
y_test_enc = np.array([label_map[l] for l in y_test])
print(f'\nLabels: {UNIFIED_LABELS}')


In [ ]:
#@title 6. Stage 1: Train Classification Head { display-mode: "form" }
head_epochs = 50 #@param {type:"integer"}

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_1 = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=head_epochs,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(
            '/content/durian_vision_head.keras',
            save_best_only=True, monitor='val_accuracy'
        ),
    ],
)

In [ ]:
#@title 7. Stage 2: Fine-Tune { display-mode: "form" }
fine_tune_epochs = 20 #@param {type:"integer"}

# Unfreeze top 30% of base
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = int(total_layers * 0.7)
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # Lower LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_2 = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=fine_tune_epochs,
    batch_size=BATCH_SIZE,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(
            '/content/durian_vision_best.keras',
            save_best_only=True, monitor='val_accuracy'
        ),
    ],
)

In [ ]:
#@title 8. Evaluate { display-mode: "form" }
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

test_loss, test_acc = model.evaluate(X_test, y_test_enc)
print(f'Test accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test_enc, y_pred))
print('\nClassification Report:')
print(classification_report(y_test_enc, y_pred, target_names=UNIFIED_LABELS))

In [ ]:
#@title 9. Export TFLite { display-mode: "form" }

# INT8
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
tflite_int8 = converter.convert()

int8_path = '/content/durian_cnn.tflite'
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)
print(f'TFLite INT8: {len(tflite_int8)/1024:.1f} KB')

# FP16
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()

fp16_path = '/content/durian_cnn_fp16.tflite'
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16)
print(f'TFLite FP16: {len(tflite_fp16)/1024:.1f} KB')

# Save labels
with open('/content/vision_labels.txt', 'w') as f:
    for label in UNIFIED_LABELS:
        f.write(label + '\n')

# Verify
interpreter = tf.lite.Interpreter(model_path=int8_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
print(f'\nTFLite input: {input_details[0]["shape"]}')
print(f'Labels: {UNIFIED_LABELS}')

In [ ]:
#@title 10. Export TF.js SavedModel (for browser deployment) { display-mode: "form" }

# Install tensorflowjs converter
!pip install -q tensorflowjs

# Export model in TF.js format (GraphModel — for tf.loadGraphModel)
tfjs_dir = '/content/durian_vision_tfjs'
print('Converting model to TF.js format...')

# Method 1: Direct save via Keras → TF.js
import subprocess
result = subprocess.run(
    ['tensorflowjs_converter',
     '--input_format=keras_saved_model',
     '--output_format=tfjs_graph_model',
     '/content/durian_vision_best.keras',
     tfjs_dir],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('Converter failed, trying alternative method...')
    # Method 2: Save as SavedModel first, then convert
    sm_dir = '/content/durian_vision_savedmodel'
    model.save(sm_dir)
    result2 = subprocess.run(
        ['tensorflowjs_converter',
         '--input_format=tf_saved_model',
         '--output_format=tfjs_graph_model',
         sm_dir,
         tfjs_dir],
        capture_output=True, text=True
    )
    if result2.returncode != 0:
        print('ERROR:', result2.stderr)
    else:
        print('TF.js model saved ✓')
else:
    print('TF.js model saved ✓')

# List output files
import os
for f in os.listdir(tfjs_dir):
    size = os.path.getsize(os.path.join(tfjs_dir, f))
    print(f'  {f}: {size/1024:.1f} KB')

# Save metadata
import json
metadata = {
    'version': '1.0',
    'created_at': str(np.datetime64('today')),
    'model_type': 'MobileNetV2-finetune',
    'classes': UNIFIED_LABELS,
    'input_shape': [1, IMG_SIZE, IMG_SIZE, 3],
    'notes': 'Phase 1 vision model — 3-class durian ripeness'
}
with open(os.path.join(tfjs_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# Copy labels.txt into tfjs dir
import shutil
shutil.copy('/content/vision_labels.txt', os.path.join(tfjs_dir, 'labels.txt'))

print(f'\nTF.js model ready at: {tfjs_dir}')
print('Upload these files to frontend/public/models/vision/ for browser deployment')

In [ ]:
#@title 11. Download All Results { display-mode: "form" }
from google.colab import files
import os

# Download TFLite files
files.download(int8_path)
files.download(fp16_path)
files.download('/content/vision_labels.txt')

# Download TF.js model files (for browser deployment)
tfjs_dir = '/content/durian_vision_tfjs'
if os.path.isdir(tfjs_dir):
    for f in os.listdir(tfjs_dir):
        filepath = os.path.join(tfjs_dir, f)
        if os.path.getsize(filepath) < 100 * 1024 * 1024:  # Skip files > 100MB
            print(f'Downloading: {f}')
            files.download(filepath)

print('\n=== Deployment Guide ===')
print('1. TF.js files → frontend/public/models/vision/')
print('   (model.json + group1-shard*.bin + metadata.json + labels.txt)')
print('2. TFLite → backend (optional, not used in current architecture)')
print('3. Git push → Render auto-deploy → App auto-detects AI model ✓')